## Imports

In [2]:
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np
import json
import faiss

## JSON embeddings

In [1]:
model_name = 'sentence-transformers/all-MiniLM-L6-v2'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

In [6]:
def get_embedding(text, tokenizer = tokenizer, model=model):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
    embeddings = outputs.last_hidden_state.mean(dim=1)
    return embeddings[0].numpy()

In [ ]:
review_path = 'reviews.json'

with open(review_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

In [ ]:
def build_embedding_text(data):
    parts = []

    if "Summary:" in data:
        parts.append(f"Summary: {data['Summary:']}")
    if "Faults:" in data:
        parts.append(f"Faults: {data['Faults:']}")
    if "General Comments:" in data:
        parts.append(f"Comments: {data['General Comments:']}")

    return "\n".join(parts)

In [ ]:
vectors = [get_embedding(build_embedding_text(json)) for json in data]
vectors = np.array(vectors).astype('float32')

In [26]:
d = len(vectors[0])
nlist = 100
m = 32
nbits = 8

quantizer = faiss.IndexFlatL2(d)
index = faiss.IndexIVFPQ(quantizer, d, nlist, m, nbits)

index.train(vectors)
index.add(vectors)

index.nprobe = 10

In [27]:
faiss.write_index(index, "review_index.ivfpq")